# 🎬 Montage automatique Locapay

Chargez une **vidéo brute face caméra** : le notebook la **transcrit**, **coupe les blancs et les hésitations**, ajoute les **sous-titres animés mot par mot** (italique fin + mot clé en gras capitales), les **recadrages**, les **écrans de couleur Locapay**, le **logo**, des **effets sonores légers** et, si vous voulez, une **musique de fond** qui baisse quand on parle. Vous récupérez un MP4 vertical 1080×1920 prêt à publier.

**Mode d'emploi**
1. Vérifiez que le GPU est activé : menu **Exécution › Modifier le type d'exécution › T4 GPU** (gratuit).
2. Menu **Exécution › Tout exécuter**.
3. À l'étape 3, cliquez sur **Choisir des fichiers** et sélectionnez votre vidéo (et votre musique si vous l'avez cochée).
4. Le téléchargement de la vidéo montée démarre tout seul à la fin.

💡 Si un mot est mal transcrit (ex. « Locapé »), corrigez-le à l'étape 5 puis relancez seulement les étapes 5 et 6 (bouton ▶ à gauche de chaque étape).

In [ ]:
#@title 1️⃣ Installation (1 à 2 minutes)
import os, subprocess, base64, gzip, sys
os.makedirs("/content/montage/polices", exist_ok=True)
subprocess.run("pip -q install -U openai-whisper", shell=True, check=True)
P = "/content/montage/polices"
for nom, url in [("serif.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/playfairdisplay/PlayfairDisplay-Italic%5Bwght%5D.ttf"),
                 ("gras.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf")]:
    subprocess.run(["wget", "-q", "-O", f"{P}/{nom}", url])
open("/content/montage/montage_auto.py", "wb").write(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/7193ZIbR5befT9FbckTXRABEECTLRIczC5HoijalMRga0YOQzC2ABTQ1Y0/VRX6hx0dMW/g6/XVhMPhHc1GrPfCYXtjfKV+k30Bv4K/75zMqqwCutmUtGZIjaqsrMyTJ0+ePHn+yvf9vS9XyyycRV64yVaLMIu/30Tej//gpdnlPPJer8bhOrz0fvzLXuM+//Z+H09uflh5o2STRd40HEfeOFzc/JCEntf4jedlSbhMx0m8zuLV0gu+PY7TdZTU9CHeHK8268ibRKk3mofLcbrH8nS1SRtZnCUoDpcxmku9xSrz1mEiv0GchXOBexovvQdSNp7f/OBFS2+WhClAWLNKlNakvSQah5MEY0Zz8yxKlmzwgYc3ZlHivVutFt4YSImXG4ULYBFsgIXyzTzaJLxchAm6lAbnq9lKq0bTaZSlgHi5IrTaZKrPFptUgMS709Vy4gUrwcIyms+jujfZjE/j5Uwh/PLNI+8sSrJ4HM69dutJ66L9tNPy1snNnzPv5o/eejOax4A1eFDBaDO7yLwo85ppktWaez7mN16sV0nmnQCmOoDOjuveKq0DCXUvPd5k8Ry/m9E6WY2jFOXn4Vm0N02Ag0mYheN5mKYYiGkkL6oD1dF8smdbX24W60sPqF6u9eU3r17bl14tgOq6/nyWhOfm8vOYuLc3wPfe3rd174u69/mbI68no657HHbdO2jtHb1F2aMnrVZrb2/vI6/3M/55CWZlzunf+5t8QHvy13sb6ZPunod/ZraH7S6WQwIA/I86LzqfPm77nvPvI9BqtAEK4ixfL4GlmQekyVqptU7RWuv5p53njyqtceJtQ/LiepXGnN0hF8JQF0LXm85XYYZWWs3DQ32xhbtjLGOgDVejMJW3l6NhJLAMDQBdL17yxY48VoodGorNQVPK9R2w8qLVxvPDzXiz9KWBs9V8s4iGhrpduNpPpEK6Wa+TeBElw+MIAwk5FHQ0Wq3mqPZNgiUhHUQbkObxZlEH6MfNZlPHHm5SNB5euA0/euyV/30E7tWYRPObP9Y9cC3hHV6UZsT5GowgSHUKsGZn0TA8C5dZCdCW+3RdQW/7sTydbJIoGq7RdBmcTvPRLZBslgWrEWjiZQpWePODtEc+U7TStt1EYFRgCnilhCJ5Rj4zjJaYyHi5/VQBnPJR0apOwTQeH4NfDNlAPsW+Ia55PI6GaZTE092PyELTyHm2tzeJpt5xdNFJZqPguKaL5ZjU10SleB3UmnO98D/yFfNJlG2SpZdt1vMoAP0Fx/24G2N5dAYg18MaWGLixcCQF2DBd+reo1rt56/0EnMUqIfYHNLhJFpvYhB9NFtEyywN7IUZCutgNP2BIg+QoQJhs/W0mn14bh41Z1EW+OerZJL6dbxd8/CwPygq8x+nBbgJzvtS0x/ULM5K1eKpl5VfFMDw8pV/7nc9LHI/9c1Ms7E0C5MMraE8csujJfu43moKHZwrwGD9o3AUz+Ps0q95MXg49s+vVstou3+Boe+v/QEASVab5STIO3KbIRgHtW3wgddmuF4DpmBRogs+MXSFS3LJ0tQFZmKwny3mF17wZTj2nq9BSt4ROsRmjf6mYYr9pHGuIoXWebXMorn30HvzKSqs0G8YFxU+Xc3DUa3pG2o3u5X+zONRU/dGLPgMk7zYM0iz9830Ev0tghoo1PM/C5PzeOlDPJkUNRYhVt0yMlXCZHH4SGuU+2hiyU6GgGkccHRDA6BfK/BvsMTHvgXk9kYUE3e1ozXu0dQdbSg6tY0kjNPIe7uB0LSIXiTJKgn859wizGxS5ikLKmCFYHQU0gIMyk7K1ixirylPG+aLfOHLrz978frF0fDL1/9elgRAGq1A9sRQY7xaLDZL0OFD81JjTs7eODtoaD2unQV6v/0FedwguutVKvYX0STeLG5/V5/ry9g1Tdf3AI5vXOdD+/z50Tcv3pZHd8c49KJeAk+vKkDk19dmwdmJSaLgLJ5EK0iJK+xgUc+3vWDPm21wP01wcwbJZLSZh6jf8/1iYX5Tml8j2XuQJsfe8SpZUdLCNliS2w1xjI9XcRqDg2LjQ1+eWTfF0jT1erfwBkvGtlpPF0pBsmZpO4srfwTJBJvRtAI9abLuXSkermv/8of/hoFP55v0uMfttuBsAlTRbNNCNspxuYbEPVxB9pkOk2i96jmUK+xX+7A4rxlkA1U9xXp9JxOu/ONOMuTaw6JarFMBso41BrExnA/BmRfrrOdMHDcmcvh7NQ72OlH5E/9BODqLRQyNLrLe5+E8dbGhrGH3HptgjzLX2B92zJlhSs7eyoNEmZnZqTTk9SWR9r65LLOUe00r91m3i6C8KHdP3ASYGWNN8CCNdrHQ1zgED7PLNQoB2RO/6ACYwBlqSOrZQTNVEvjXm99faG5L3fSvjAzUtZe4Om/yggxL5BQWyBXFlaVWwC/uXDGCpc799a2jyaWwIJV+UpW8aoNrld6MgJZaulMiylbJ+NgtcJkDZpPS7ngzCX2Rxli5ydsmRo4jRAzpYc7NPQKeUHG9yfdTebenZbLhK40IgDlTVUZc4s4F93b2W6Vq/1/+8x//7//+T573JhRFxMs3v/O6bPfmT2h4X7j/vsftO065qa5B1TgKLUUbsMB5IaX24IdxnNaanl9BpP98nMVn0TueWLThFxeou5H18xsPSyCeUt2AnkjN3mQ/cp9/84hvNR3yNuPt2W1pz12enx5zvCQcHFnyIdgNA6onu0ChdcLhMfGugM9rLlZziEO7lt9C+JwMpXq+Hs0yxI+RMH/WIttNcvdYed503T7sBTkpkI5qtzT3vlVYp05gtEojd1F+CLPd2/vixdGrb55/8+rrr45EpMBhm+R2HG34gzu9VXHheCE/i4UU6l+cyeXHvBgdQ344+ubrNzzO8BTh+5hEbN5zqr1ISJulKvJArJjmCekQ0twCkqI3WeF4DkQtY6jmEu8EZLXx4rlHTZgHklpi+JAz8Ceep1Ka7i1QKUJXJGYsxtibb/A/9w7+Sb1LavuobkQ/+D/jVerd/H2ILryx9z1qU/n2/Qb/4W2AAHhu/pkwhVSphfjdXOzxmJ7yWQp84tCOm8WCDf05o7owpo4rPIMCg5LNO481Mc41SmXFZcnNn/g34p/VGkdoAEh5R+SglBohUjQVOXsT3o2P0coZdYQLDDzD/+gdWPKyENU9ds2OqYPDsQz0dSZ/lysiCOUx9xngBkoDtAFlCUpGccSmNtleRhzyKtK/KU7tYEIrIGOJbs5WMRQV8oOJwxV+4nHMwaIPDDNOsP9CrTUV0h7vC3b2MVPfb8xfTs7eyT4Qs5Snk31MPf+AGPbRBca+iDE12iD0Lh70K8SRqFk8pVuPJ0I+i5ZjaKGoCD4BNJPoJPTOcLYgHe+dbLCNe4ubP2O0C7AP9E7WKsPHq8S0zOw62mT8cwG06h9AFfP/JAKVNtM19pIA2oXPn7/6Lfb0n0LApBdDMSS7iZ3XHdNdEN4e2luS2E/eT+7blDz+IALZswRyySUD4gLOuH4yl0KFYnMKcmcqp8oSwvS4AD3hIsjMBmWYUBI1oUMOEr//H787v/njzX+5+a83P9z86ebPN/9w8483//3mf9z8z5t/vvnLzf+5+fv9f/nD3zUGZCH4PwMDP48SNN6EcDyHuQAb3R/+jk/3fdshFAai3U+g90qgUBP9Ic8OKYWsvKQ4iTiF2Mr251Sl3vzQMyrVZyQvqPowxk3Sc67TfXveyLizk01e5wqgtayAWIcqCMFgn323HPglIMjVfHf3hjTg93y+KC2UVSohVJ3oRp6YVlEZ+rAtXVBolUTbOhkBti+zkteqUT8zKimW4qnodaT21mFe1C+8B5MoK74WBJ2Pi3dOUUO6gybo3IrypotTVq/0oarE5ZgtY5zULkAmIP7qzb/662dd7OyDB/8Gw662Zzf7STyGuI0NuqdjPR1AbxhIk/3WQJQy0rwKYn6taAHD2a1vInNU6uKpDHJonATcb0V+GNG6AWOJtEqFajjfcdSliicin5D3yAZEbWjOujQpQd6SppU39G3DA0ipetSV3WtF6WOTFKddsIOziJylp03bBehOo62zNZNm6ixODPQihPI90llEei+jn9i3TfYb7YFDZm6x96BXatZoieOJaEshbQbnNap0C3ncvq1QZSvofOrUXSqdpZtFIK/XDNodwoOZaC1NvAMJ22bq2psD3gTNBDT7NdhCzfuYbz3UnnaRgWpPz432VFWYGZWVRm9qSmhFDC+CVrP1GKCh1Vbzaadm6q1Rr91sXRd0lhEzk1sITDAGzCjQ6/etgvMSoa4thVvqNq2S58Qwx9zFB58bxik7Gc4FWaQmTIc7StWGx9PPzQ9i912sZDuF6A1kRAthyrInCnUvf/wnaOmx/Z14AVtdjccbNLdEHXO8wApSfU6k2yJrrY9hQohq3rPt7kQYZ2/YWnELKeVkRXGu0j8OHJQXUh44/lbkiL8lAHyLu1gsIiAhrHEfo6yj5qPfLfNGZFjsLjdIFcMKuPHCBEJxTk1GlKW441JpVVJECakrR6qVOaTS7jgGi4qk1liej/m8skX0ZU7HRv8ObESRr1piRb9frAN5PWWVSWDarnun0SWOLYvRBKJt12vwXDXu+zGYp7v11DFLPc95WLc3JxW2TdYQtLxfozb/nHi/5tlRaKtW2XKMZTwqVBnrcKmquRQmnROsnPZgz9X456WoVXBbM3gRA0UaAHRsCvSOs4s/MHfgOrBh8Naso6KUe0WtjKctNJc24jvw/Z4xCuI5pZn0lclJXvbACyWCC5mlrBgWSVSHVcsFKFEjFLgh1ohlaXxrOpbvAWlFO++pbNZ1r5UX42AMSE/dCexWhYlTzMaS85xXkf1BZQiZMda4MCguxoddbxYFSwoXPUXJtjTiUsNpV3oabFUi93/wr0UM7r9TdrMsFZONdndBlAsKAvqg9kzfbpdoGUMjZ99tuZLduqKcdmbAGk48v3kChYwRn0QJqsNPSTGiiDTMvs9NwS9zmV/AKnoGb5mJmOHlfIPDayreEEcvfvfq9fDTr7/6/NXzrz59ob4Fdhc7iTKK3TLG8oDuXhQlEUaXgFBbJlSOCo5mYmAlfggXl2m+uaXRJp73KtAV29xn1m0Ih7eykQkymW49KQ9vaTwTWWisJywefGGNV81WvhEqLgAX1bcJRLcAK28ah9zhrIoKp2Gygy69nGKsQih8k8gYScxhai4+Ppg+Z9NLaCOjTHjzg+x+AU0dcpxVIMVFIIH+D/u0HnbNUbnLQyJUTfADkR18FiYTauNC3CTL+OZPibO3Tik5TlTy9AJcjHj+zfsIR0A+/AugI5STN5ZgQp2g3eEMvrCJi8GFzMURyrClCCFG8PeJkjCLqhyGXBYeJmXZPxFVqqwrPOR6HVRZEhcD1rmgKTqTRQ7FIsiPbLwktEJYshKrVt06MuRrOuGJIpJW885bA+UoaL3d2XGaMuO3zACvgBFYXBT7wSaxNeJcLi8NfEc7e7tPWKfYlScXZayatx34zkWi0O10UDhL4M2BoylejaTauZrocykcLzvm/rKR3xkUCN0K5WypBjmaO4S5oRAqHQhv4quFoUYWRGpebuuBWc688oKIr7/GbB48vrPNYhfDMk2yKC0jnSgmiA7wwklkhaJ9YROcdQvOb3raZ+XQrW3bmfGdpS7rWTSC7or+65pf232UWOIgwC2ujS0Lpw8eLNJiy/Ih42tRsW/5mDCU4e9uVbAv7JJmWrtLnOsu4cwljyAcMmrxB3dmSCgwV7c0rowD1Yg/W5UmmXLRtSHVj5RnpQ7TSjF/LochcyL6QnoAgEuJ3ou4P8g5LxSpFIYiVTCoZ2E8ncLbwJpdjiIcUsBgvwwzKGISd2WE5XUB7DuzCe1gL9+Ywr7BXVmyJbGdhDWQR+c9AhV7HHmiHc1EAGqL+PJosCU/jfoy2+Qr6JWzTHbSebzNTUZJFJ6WSk9GDsyjHTCLEBVT7VFBS6BmhRMojU7gtpJwA684DQGqTTZUlm6ESxm7uRypnHcy6nfNI5HkTsItbkwAZPUcyhu2IRQdiFOT29H2qIEUS5EDu1imvtaO5FhHfWy+OXtXo/7+cn9w7VvZ123+PhZqZR9Tv7T3dc3etquj2i6YzergMUXMLHfMZumQ/+nXr7/+6isYh3///PWrz0SSIePyv/rxnyjLfiZbMK8+B6Xg5xvRF9FeBdmDCq0aSz+1YgZvbv5QyCa8f6mg4eqo8IX2rbg0h2ATpcNCrKvsH1ZO66/7YFngQqrkWBsypr7RKRN61rJ1Tp8VhuLLVOGx8KEBWbKwYzMPV9pYu9WiPqa/z1r7cELzflV1p8ntmj6Wmu/9+L8s21s7NKRio1tilSECop03uQEvGwycDUgRYTE1pDUB4mkJLzy2XdQKz0XVSFvxlO5U6Iy4b/EP7Cb6Y1TYLIh26Kqd2VApNpesdLp2qGlE7w/dyQXltfm+GnND8Y/oFl60dSv/qTWpoAdq1YwyUrVDxtE0iC6aFIzhYS5S8RQyMuxiXIsZFeORmi02y1z6xYqZ0lvcKjAjUWBS3twkrA9ziKlaCI5bcs362AzVKvEqCDAm+oJ9aGWo86iqCuSO2sS+SCkgpCe1fvdJIQ4Yda0iA4cE61gpL/YPBkYVXzeTrsWHg0rBJ9uKEG3xPbuFqZtWnDS3NI1CItyZ18f5chNFI+/NUtMNWHjOde293ZZPYmTgtoRN2v1kGyw+AFiqulI50vG65buUTQbvAeBO5fSdCur/H+j5AF20q4+uqKJzWMpevvdTSOdK6XwEPNTYFm9RUP9kJbVVTfNhBTmOTvqjXBizzulcwmJMgZYzSOnLkcSQ72pgNCR/kfQyPSeDM4kBvljXDmMVRJfXsyMvfTDZlfbVn6nloKe+xOmoEtdozeF2CiE/UUd+w5nFcb7uve3m4R8VP/Cd6l5R3NiTan58bUeNg9wE87a5O/ahqlLqL7abFpXstmJIDTmF8c1sW36Dp7qSMqU2cA1GZROe+uv+PpxvSt66YgRezcUAn0GPKryeuwxY/5kEdlknn1tMgwYfPTGeLMyZx+DmAQi19SQ/WiBYDCEQS7Wvm26D1dIzqpw8AgzIzWNAaiZwYhlV3PPvo5fQ18Qby06WFOnpDCuVnjQohrT8tlmEnZTW1M4XembYz9zH7L+QfuNiaW7rH+WtYumDT0jDXQOnHiSLIj1HSvtwsIsH18WMvBP2xOaKHt5BbMqnBDNQZ4ntg0h423SiYmrueyN9D9KYWSHv3GE/KN6k2tZAsanMzC6QMBmsx6kw0AHnKFHMjSytVA5spQo6HLeobiCGuiYv1kkgH3xnb+6YB7xmZ+FdmdLwpBgQaO1dmdakWq0cEyLcLe+13JFhg32fL8pg3sXusiBrgjZiV5hi3awP8hU4z9N2IY4J8OIQfidORzQpm5Czy7HaLdqc+LdNxhzVabt8ZO8U1YZlacfvnT8eu3Xinonqahd65fxeGA3adTlEonbVnrQQh7e6GPoVMZMLHL/BSQfkHXnJts4wX8YheMTbZjk+S/bFTx5bItPT+sKq/vBo+9g6Fgk2UICUkT0wr1Bt1dnWFRJdlmT6IXWmGemP+BkbK8HOegqQ2fNypF8Bo136hPjUJMEa7nOCqOXhLPZj71dqLuKd9WwFMQb0IqlV9Ids02wAGsTWKvZv2XdZwdFW9v2VEGL2TOWGtUXZWubZxC8mawKaubQ6S0iPwsSdo4hUMI9+hipZNq7qFti1gXb2tKICy9ZxqEouW3ppB8b3qHtvE8U/XNW9HMP51Yib1vrWP3EtbSdixwQcjEiOlsEup5g7FOYHrS2NednN4/1KcyEyAgB1zyNx0xSgoeF6ZHTqhm/9Vc+0pvc/TQnffFL7qdPyfi38jnYM7RqnTWxnsRx9taacc3kQjkBDGqMCpwQKIz/+pU7/hCizV2jgx7+IgadLXz4SI2Poi4ZE65mT+ywumCERbCCrkdE5CJjBD4cM1Tzuz2KsmuKmbFxXW682VyMuHemQqJ4J/RnaQu/G4bGK7Xv4Bm01t6uNUiW7tZnFsLzQSd9NKfnbtlpa0IttkK2JDaFVoZflRRPe7VAMMXB01lyv1uKryEcvy3vZLN5harET8WGmlnfc9ETaMxp/HVhRARIW3dRzc0BxgL7ILIeozGsxdJCxlhkzfk4sovZqR0+LbTZeGhEN7dZdW4Ns7Ie5VFQsGiBbBih0I8Rybszn8CeoWcX7eRlL5+kOdhEbZ1DzvtWYiVe4VRuXOhCO4qxfkU0oyAlQtX5b3WIEwm0r08tCRjbS73maG17EfcxYXMR5z0fr3DsZEuBLADzu2N516QiyI7dDsIDeDQH9k6g49+SLkLWSzGBjacVqdax529yKta8VXns5Sl4KHg4LgsvwfsaQ/ZdCLUYN8jI/ZzjUQ9xw8uKCLpWndOq27YbXEX2phMn/mo3Gg0IToXdqvvg1g9dbg7KXC2ljuW7OsabW9MfliquTlNDDsqYjIauR+emzE9cAGVdsdoVnjrRdCYUeU5pogVLHENcCoKCB2zKhwUyU6GnPGbJggV6xcKALR2kQcxtUDuiYHmMrAVUkCm3yPsaaUSRaRS4vfankeRVDWEPngYvSBy66RWrkmMYV1xTCZomZfTirU8lGBmwD7N828zwUWKLbpZ1aReyuK5aK1WvcxwQlFe6p4Or6oABoAYAI/ivE5u+5apl83anaRHgeb+WXfpEUKqm74y/uzZJByUsxWUJ8FJslfstmAF+0YfaZnCzzZAY2SjVNxh3ohUS0xXIxwygJtH0DQdnrSyVYVc0ckuYzcFWVb8uoyD0uRRZGZpWy+GseC0v6+YkJqBaz7qHMshDAxLcg38J5TvLBnIVJLOoipmqBjtUMOEsunYMmJixPn4IQp03E8KxKU46XqvuuvI8NPxvmPQ1Hl8NluECElC1x1BYX42idIS6MP3hSkXuZQaWCx+ne3e+Zajvgl/EW4Ov8x6xH1YP1054ajKhDcaKu9FOeFkaj1YVRj5scIuSZB3oEglaVmSoSTO5c9MMf8znfbvDoYwvUprHAorc4bi6j88B/DWkoYBtYioZV5YltmvwTyEu1JvsPAnbdIIh6wQ4e4U07BAy7N6VT+Hzeaz827dle5ReOwjz7B07KnOZLCfMJl7+db5Lgqd1iFmVA37787XMHVu5R5r8aZOl40URwajhfH4cGYKUuEU7u2cquwaOB24Z+y7CDzmO4WJf+5CNqCoBDOhgwBU4kzbuLMV7UdfYS7UomEQSjyXzeyMoynF7W2pCBe8Mh/e/R/1vXjzyiKwrWDXD98fi4tpM1LGVgrlvvMTnMKm0yyhuqbhiu0mBMSWp8jL3cjZ4FSQ/J1m0n/sMjSSDx8DV2ljC5fMg1kD48omJHfJXD+cOX0SqZxaH3ivmtxs0sm/p3RfT6Dzdp8jDFYS16OJXW7JJ6yFCqs83Dz/Dz+80Rc700nEarcI4+EM7nYBdz77er+eSXgxGMvJG36AZOz6fNtTgF8mJkLhJbkhB26G+drDb14l5T2dTtdNiLUbn5MfIPmPCfnHSohFCqkcRoOXcqh8UE5YflyBgjHhddbDmWaM41uiWP/B2utfmLjIDp5TtHjgm7dfgvLmDuJfL8HCu1ihNsqbfxB/eWOr19KSHLhkjzHtPaPdxu79/J7a1bK77blE0lAysH8yWtwaMCyCBv6yankWN9fwOLdgNp1sYbDf+zybY8EznDg8FxSPO8qAMk8RV0TdAVavyh5jih4R8+UDf/mBUWcTlycsZfOvIIVtK2W9uskMCMJ0UZVTznkLxmVs1a5yUPOINt3+6dLeT2dem8b05kYmDEWc/oOB0fs4raavfsaTwnj5Lva67frepvi/NfPyiiB+vNZ12NgqFh+Jx+2MbMBXZDrwxRjuZP2M12sxpZei+w1M1rB2gFrubywlxcQWTAdT1TqtFDXMzKr/M8lm7rD6UV6w6hzW9jVPyoheSHOfEPxSktCJgO7wmFh+DwSd37hNFIwWNcHcrVI1w97tRqu3M2xUp0gSNE5UtB9Ktp5nY3FW8KYUN6zrMwcWfL0tpdPlkyBHJJ1szHuls5GKVrE6DSanaeQPDK+9lZfX5uNKcxz8kdVzezSKkvZXsfq0JMShrVSErXMfCcZ4FvxU7w5LB765C2T4X5JNsjHDvD8fu8LsoAAe6gAlydsFVOgTDvJJJA7lH5VHxsRpnYISb2mK+DlNfMME3p1kAvDV6/QEXuftsJBPkSunpYNWwI2HY882MBPO9/G01UVgbfUvqCu8XDXVYSNhjkgto5JcnjmiUO9HQLxZJROgiuo6eGNnHJ6M/5MW7RkC1koNLOhi5ozjg/V+rYqnHJp2jLoPUO66D1Tiq4yhZDydnz9spXwpaF/BRKuSdQpnyC9XzY2rFg37tYfe6DRsTQBWlX3Z0rrrTaDvPVto22X2ad3XONba+v+bFRq+1cSaWqF3Wh9BIBIoeo/LLoFvr+MHL8RUhxNxmCYqR1Ub3IlXVUjOBdkzsqXhglFJFyQWVcqyZW3NJZiNqzgH8QffPxx95BHiiKOU6GzJsbUc96FmJtyyAAPrBgulAffVgKqwd0PcVTNhOQ1jg8pmMxKT7piI65TcYdqdWTQIqpmfTRbh7m2k+8QuX906dPnchi4H5p57qtKlSc+c7jCfjSx3iFxF1+huwVs+PMPHTSe3WIvgVyCKTxuygItOWayS3b/O2r16++evH8bfGGzEbRWQOwKPE8U47g9NVgS0pQDwSM9iPSPhEdGRjKzMJCY4ceYuTt7ccd6O/Wl4GcxTvFYZw3WOMQOZkYGH45fg3sm/0areOZpG8NzgBEWDMA6LRunZbRFiQD1r4woZ+XtV8iq2YlyXGQXi6zY3gMMfMQFugwWc5UiZww1mjRBBmGm3nG8uATq88ZTkZVR9x2i4QbXFBt2srriWVtyLClKDABV1NQ2BRU8X2v3bRBPDaaUqrQCWSsMMyi1UI12fatJZ6qtyoev0PQbxoYRRdcfyVxrujYscqeiScfCBoQfb9Xcu2wsZBlZRyVR0yx3ExlteJiHaNoOqYl6aF35JDgMaiLnRFcPm1I7w3x6CMEeUUBBzQ5xQO+9EwqmgI+lNGwiV7x4lYUejplkrrVKj2mM0+v1YSqZR2PcUFpctrqdR4LenqPkWIZOIrp3thrtA8c9JKC8DK6xUCeeVnVUlD3BLu6JpZn+vgcoQtQ3FDnGo9BjhnwgCvhUZ0mCVTWUkZluVzxoTzN+UcqJ0OHDEhJTB+GEHqkbKKJKZwHkgNUxkAnS/Rf8vFm64CGCntEu5GrcCODSvgpa5MYdcg1QQuKMFAHdbQq5jh5VMEJeGC7gpTQ0ofM+l5OH4e0djwVv3jUQlB70CBCQGvtJ8Vg8YgE1NFaQkL4HW8W3JSn2mZtuwmGVxnWVHnSalulmoOPKjoqaKDPSIEASEcQFopMrHbsmKT3D91Ghu0YF4k4q44lENuVGQoTMOP6FKyLj+TIVtva0mH0mJaNHv32QfsJybz99JPH/O0cHnwCP9mnj9s2VDj1Pu55u9B1cE9kNdpPS5iaRUtQezIExoLoDDeSgatuHY1VMV/BYCAPqXmuWUy+LvOmZ97bZAe3Qmep5gOWdQ3zSbHIaY1ZrU0Rrmh/MTNoCu3tteuRA3PtgsaVMiaLkTjsDtG0CDiYkqoJCILHFwPHDTzNRwf0opoZm6v1SCkSdCuBT9w58LffSLvwLmMrRVRgZE3dOF4a7xhUdQQBjT0PXKbUEE4nfyJAkpoVwqS3NHDp8bbR1hATjEGOvQah+jBHu76fd/a6n3YjcS4kvF15OlBCTr+nN0IzJ2HAVZNZvO8rD/QVJVKzrsZz+p2vyffGp0H/NdRZicTp9LtEtFKRIBmljXaeReg8hoTDrP5NZtLNjUP++Qguw3CBOXc8H2gPWhrZIw06gFmKUuTDE0mpKEIQiaEOd1aR5RF+5JE8hGU0pah20Pnk8JNaE0k4advBAAAtEn43s9XoEsE8wS8hkSADyWSj4oJRIYRT5jyPTLY/Q7c264z92kEz2SwDGGunDEKVEJ/GmSS3o0+03EKrCD/oIZyso3AhqQnOui19crw6HzJbVsyUBXdpvPXlniCwrlJlV8uGKYAbMjNuL1mpc7h9Apf1VAuju1tvrKaEh593YHJejhbzj0xXYF3RECOGaGnyhUpCQfGhAxFNbB4Dijh8W3IpphJi2fftgOmU4prNJjj7chw8/+JaR2MNpKssX66SdXNCNjIx/nt2pENG9+xOj85cWRYPkjIrnVTMMNKDZJGUrOMpQMhfsExdK/EveRpFFoWAGEW/V9c1vbfIhSmLNVv5WYV8Hg9r4A/tJy0yhKcOmzKIAIc9d/cIFptNQE3XMhMl6rN2UQX9Q4iwRGrwcqHAk/XQj478Hll8LZmM07Peutf6YErJE4vZOLfhZMNvRvziyyvcOWKzhJAzKbq413L4ueN0J4zfWZBl8QvEpMDYYOJRNEWrJi5SHyPsLdYKqVXgxZeMNcqIHhwxDVdUQ8x7VPA7AYSQjBfMSLHVIiP5GNaB0HjqrOX7MSbcb55Hxrhf/mnumcAX7Ri+jXneqHzn0Ohc+QQMNIo1EyGyj+ypkn1wM994z/SZwEn2skmRKlLzS+KBpmb61FpVpCuT/IIe7/rhnu3P3/A7MF78kJY+ZkHM5FMaK3wUxjv94h1jlueShypEAqnMhD5KrsrlPr5MI96e4qTMc4KkvtDsT2agkvMT9iLM9kY+sTPKc0jlHwiwkTT6yj2iaXQWJFukmaLEBtBMVjDhR+IKLFPL8NembdpEVKaQNU9hXEvSwFSHEEMb83B16iSs1hDN7YgYd8JksmpVf2vtzRHqVIjolViYEVorno15NImyMxYKE61VXBO1RUZ8UL0G9adGsSMwU6UrynPaBAE0knJJTMyTu4kGqRIUsoX6qf+lIagr8aC89oIra5/HF3Ei8W8xQ7qm766SJOYDDr8m76lmXdEwKJC6m9h4rSp726KEKudzg8DnuNs6mFw3F+tHzkvjhfjsgREu1tFMGBw+xDKHWD0vs8NLZX5keWgLQ+42D6bXUphpWY6M4kns24NFofLlAukVTLo6hTZ4terGQ0AfENLGVPNSn03je/Q+9cPlZj6Hh1Yv6V0dvb3ujuc95lyPVq7rYt742VReSsfMq3j17XX36ovr7pTThkT58Yx212HI709AZcQdDnx/nFD7WB8jvMW+UZ8iR/MVPhZ1XRcRNem162ZnvNycPeq01jt2Cr8RaufwP6BUSwAAcD00bxrZdzgPL7H+UzuKXQ0twrUEhXetRJgXhCiQAw5TEEmgelvKdrUy7sqWCPfCiw4+D8J2aFiMJGEAeOAls9dL8TiRSWkf3tKO7JxhOJbKI70F2zyV+5AExuhoSOtmg2UkPVsmSTuUU969MWlCXePTSoZ8w3SsWpy3JfrSjaq6WFkGFboc4XmY1xAIw4ak1CjiZNO5bZ358pifOvOdI46cbuQJDzcYGJMJT/BJtZ6/yaaNJ3ramZbDtNR3UMdS9Y2TowzC/+lkv8/8BOsiSe2+pKj97rt9pKkl697/bmlg2SH+3GvVq9iCrGahTncaToukA7LMdGy7M7s0xvr6WhtbrM6mc5V3/QekIM26b+WIwfac6qfgmskCHDAqUA1LEnS8QwE2darvmGEzu1WzhPZolbn4LB3ymiO7ep5BeGE1BTaoGYo/qCB5XoaMLU8n8RmSZQQ0Sh4ctvgPDxfbDw/No3T7kTRZksL9q+NuqzMBK1mY31R+61ewAZGJ20ybkXwahZAb39OSFqcgveJgfQ/aozdHu0SJs8KL1frNDnZmCy47ws5sYHl9+4H1+WZM8nZG4Txph+a9cW9gBwp3xMdVI6DcZQJ8nl5/t7xyZjhEwo8Gvmbolo1qrLTv7dvkRvvn+wPXgrpPCypzhXy35JLKk/DZb0Mt6QxpP60iA3XCtgs52XyNrqfSBhbkzGQPNYL/thxtDKxAsHEbDIxeo+REVJmgkkuRmmLox7/Lc9tNgGsj6bQ1dX4e1M2t+kmXAzJL3s17xbfgNE6ppzezRA0XedCERKC7n36TEISKt2K5hpuOY5a7ggp9VypiaEvK6MZHFDMlL8xnTW7SwXyWe+I6MmEZ6FlusDugzcEx9R2IhhwVjBHuIa9F8VCr5Wa910gD+B++PnJap/XmE3wnk6ZnU19fNc0w3OxQt2TIB2WoLPYcqKxHhwEKzX8YTNLuRGiKzcj0aVBHaKfbCWlueAfNQgkhSdIkipNQd8uwmjYZlmBuS62F8skHjXHZEd/CjwNwMTJYEluvmmu0Ft0OZkV2q5YECs/yTByFn1rtVp617c9WZJozMPHMv0kiE2CjUNRt0n8kh+JSReqBuiw9np7m+xjGsaTLR+TdeHXG74T8hcF4sDrK5xJRXbexc37VlPai3XrA3BzBCfkWc8gXvuDvsT47wjdIj74wW5JY8MZRPA9Szj3tAKKk7xhzqvP8uPxcv4xJ5TiyUhVyqkq8RxRgjyjzco/ujeLxBv/7xZp1vgGZTzwaAyP069H3PTJiOLJmsH62DrvIEGQUQbx/0p2Fi0UIu97TJ3XY/lfJKGQehaiH/Zs2ou4o7TXkIln0JMp+tJCCjgLAydGPbTmSzBvhAPeQZWKrcRFlD2Was2ku3cBBXJ7dQ2EFKTi+GE4XIguBlXZUNJZMsKKo6bnQvXrzwvBXdbNB7jQ6nRzR++nAflUzE1uJ5t+jetyPFzMbElb4+arTjdKkhKt3K+GWbMnkLMGBFL/ljXK0mYohWpBolUo4vECy2eFtYxY6XmKGOq1xn8AkF4hSjlrTqPscaqctKJ02gIRBzuiZnQ/ATOn3DbZOz39dECB4lOsccib0Yav0qQIjWTntKlqzxXp4VlZ0CBnIKdlyH0TLir7gpxCdI0AXJLaThO6jKDWn72+vL3DIlHbs2QlruGaJHLRYv/v4do++dp7vnlRhz0+zikldAvFy9wpYjsritLEKEXRs0SclBf3ScWQYOYQuCvQlmBnecs90K1rxRTypuCfLwjhxAlbNzmW+hol9SROAa2QVpMtVZozv22R5sk3QayMBMVK+bC4koCa+C45YKxOxtdqKl1/MxOfGXdo5isRIJfipumcmYk50gz4ogtc19MPsmOrgw8xjGPoI5CEi5dZ6lLbcxKXiZTm73VlbXXp2B+PwM9n0+dVWGYvS5iFUImlq93G7h31urNEKeYgbj+Mi6yBIrl5N/7DtzvZOES8JKgZis2y2TNYWmjDZQU1cwFoH20M75xmKX9hDn+90+3237TDXcpzejr41oZRj5s7q0GZTJ49nAcj6slT3C/E3fMyHx07tL6SgdjumMR+56GdQbP3HPv0d/sfcry56wQW43iX+v5DA1XPe8AJSYFUk3nJSeP8ZrwgIcGSwVjunMCuL7falVN9d69qnLo2Fi+HtPs7v8RFUOskkITC8MR5t72GFWF9Nzvu26X60WorQ2m96hSDLonx93LIYdsFXdKq+tJzfDmOVFVzbvgIN15g7fHvvsdjG7VtWmB2+nh/u/Hry7cOQl+vGjzUvMKcL49UafFF+qMcQ8/TuzUY5o3BuRYeLDbP3NmVTMad541Do0rNfsti78bXF4VpTScBuyq0Sq/CxpCdsVXOk6GcAPWXI3tXy+uHVcnQNMK80Y+iSY2LJr2q3fJ8zVz9t0Rt7h7/QdrdGPcVtbWmioSpDH8+RX50OmXn5ecjUhs9yce4U8lngpg8o/YMdbRKvtOUz9wPiKtOL0+TQOk3+Fb8ITSuRE3o1WiUadGLita8yZ9MZOptOkW5fcv84if/yJ9pWJfvilvbIbMPb2qHV3ZTM8eUe0dZdBu5ADmWMNZVAf/a+k6KEMW3N43ZI1KDatP0oyrwSlz+DfC9ZM6w7ft/2Yr8T0e+aY72kupkLk3iIwNxBv3s42NbKjefvx+TOztIdiXs+CLP04SqjVSiLZhQH7bqwwSEKH696HkoPhm1ipOD1VZbCUQASP/OtN5njuuY4raHAkLxNh6bzqNH/g9wESV4WjsBct3M9nb43ZdpEk4jplveuCLrPGy/ySPdhYBmEMNMvelfv+vvh/kCsUV3ejMxNHcaibA37zZtvjhpH3zx/+w0u6v5uLglj0DScRL0MdqbuRM7EdVuCCYZHTu/KZPmY6IYMp1jpUiof9EOoPgfOdq9ocCB2K/B46D+zWY45NDEgPJMEyKZYG7D16Y8+NaaB3rJ3RaJV/F13z3rABgxft8CrRECYO48dmDuPS7gAAhC40AM9QWFC7+ArmXrF5Hy1mdDDtveKnq/db970Gp3u67fPe230WjaiVdq806SGGWLi2F6nfzbon50OnpVf7rc5yXe3cLaag5zoyvy4n1r0yvcLlfNUtBG8AS3nLNmoiAsatK+qkdL4Yc1XsgT9RjvXb5gXHUZhpuhZv5OTZqvrInEnOX6QCXKLeKe+Gf/V26ZeDQ1g1/UKPR/eRh56OoSBoFlQB677i1Z5PsycoJhTRVcuAGm+m4Cdv5cd4wfUo4unq0bbw26Y0VURiQ26Cfx8gIPeweNWf7GzbVBBOsCzcBFfAO41R37QVdduSOTAp3V36k3hEJHVUbyAqJL05BcdP3naDzG+nBBKUp4qz54RfLVVx8vTZ9qp22Pnp/WYqznCMoMVgaC5eBT6P8M0+LGhS9FxSAIICfGY0xPKmzqGZwPMbRZCY8MvqHKnYfhOO7IMcZfx8CfaPGOr03BvwvqWObOwVt8ytg+3dxYilAHAdM3drnt7xhQYT+iedYaIhq3cKF8fiQvKLZlRdplDsWq4biUVMf+Ynmf5HpvLSnvloOSp/5lmYjdB6l3vSl7Yl9nlJtieXkMfHVxZbUx/XzQokoXf5ufZWoVXxedbJEvajHzyor8vkth+SUhDM9umhZrz3TptYDvVsUZP2nhuDVmHtH5hBUGxzjgfPdnORXZhRS0n9ht+bNxX8fF5hcpCNPB1CLvsEfkXFctB38W+7XlXGDwO4kDoYbMzvU5RktsuJV3l9RV6tt+TsB/4guHS7OOa6R+zzbztkltnOBQP8yFylMJDbmikf/MhFGTThURgvjwRUu1lS5rPk9mGLvhveJcYB8UQQQmTyTA0zwJfdZ9MQVp9onTn3/JeoyEYZXap+brn/9sjfMJCUmeXP2gKHzt+Ntp8B6t2R2u6E+2EpEGWQAWtCQvjl1Vva0fyd5Sr7myReT12tUgMItCEOGP91ODtLXOEGJtx4Fo1e6GoK6xNV/OHoNDkESllEUExL2pupvDcjzoQ3XXY1C8o6hd8pY6G05uvlCEwKmw6hilJ4du7K0u58cmzbzGHzV5xqN7iJxY4sZqbl3K7eZ1fFa7YywGlXhUn5JLvgzUei/BGxahtosbvlMqhIrFeOQqT//W/84ueanv/DwQLuLxzkgAA")))
open("/content/montage/logo_locapay.png", "wb").write(base64.b64decode("..."))
sys.path.insert(0, "/content/montage")
import importlib, montage_auto as M; importlib.reload(M)
import torch
print("✅ Installation terminée —", "GPU actif 🚀" if torch.cuda.is_available() else "⚠️ pas de GPU (ça marchera, mais plus lentement)")

In [ ]:
#@title 2️⃣ Réglages
#@markdown **Couleurs des écrans de couleur et de l'écran de fin**
couleur_1 = "#2E2C51" #@param {type:"string"}
couleur_2 = "#0AC2A4" #@param {type:"string"}
ecrans_de_couleur = 2 #@param {type:"slider", min:0, max:4, step:1}
#@markdown **Sous-titres et découpe**
position_sous_titres = 0.66 #@param {type:"slider", min:0.40, max:0.85, step:0.01}
couper_les_hesitations = True #@param {type:"boolean"}
pause_max_conservee = 0.45 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
#@markdown **Son**
effets_sonores = "léger" #@param ["léger", "aucun"]
ajouter_une_musique = False #@param {type:"boolean"}
volume_musique = 0.18 #@param {type:"slider", min:0.05, max:0.50, step:0.01}
#@markdown **Logo** (cochez « autre_logo » pour charger un PNG à la place du logo Locapay)
logo = True #@param {type:"boolean"}
autre_logo = False #@param {type:"boolean"}
#@markdown **Transcription** — mots rares à bien reconnaître, séparés par des virgules
vocabulaire = "Locapay, démarcheurs, loyer, propriétaire, locataire, Cotonou, Bénin" #@param {type:"string"}
modele = "turbo" #@param ["turbo", "large-v3", "medium", "small"]
#@markdown **Vidéo déjà sur Google Drive ?** Indiquez son chemin (ex. `MyDrive/Videos/rush.mp4`), sinon laissez vide.
video_sur_drive = "" #@param {type:"string"}

R = M.Reglages(couleur_1=couleur_1, couleur_2=couleur_2, nb_ecrans_couleur=ecrans_de_couleur,
               position_sous_titres=position_sous_titres, supprimer_hesitations=couper_les_hesitations,
               pause_max=pause_max_conservee, effets_sonores=effets_sonores, volume_musique=volume_musique,
               logo_en_coin=logo, fichier_logo="/content/montage/logo_locapay.png" if logo else "",
               police_serif="/content/montage/polices/serif.ttf", police_grasse="/content/montage/polices/gras.ttf")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3️⃣ Charger la vidéo (et la musique)
from google.colab import files
import shutil
os.makedirs("/content/montage/entree", exist_ok=True)
EXT_V = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
if video_sur_drive.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = os.path.join("/content/drive", video_sur_drive.strip().lstrip("/"))
    assert os.path.exists(VIDEO), f"Fichier introuvable : {VIDEO}"
else:
    print("👉 Sélectionnez la vidéo brute :")
    up = files.upload()
    vids = [n for n in up if n.lower().endswith(EXT_V)]
    assert vids, "Aucune vidéo reconnue (formats : mp4, mov, m4v, avi, mkv, webm)."
    VIDEO = f"/content/montage/entree/{vids[0]}"
    shutil.move(vids[0], VIDEO)
MUSIQUE = None
if ajouter_une_musique:
    print("👉 Sélectionnez la musique (mp3, wav, m4a…) :")
    up = files.upload()
    if up:
        n = list(up)[0]; MUSIQUE = f"/content/montage/entree/{n}"; shutil.move(n, MUSIQUE)
if logo and autre_logo:
    print("👉 Sélectionnez votre logo (PNG, fond transparent de préférence) :")
    up = files.upload()
    if up:
        n = list(up)[0]; R.fichier_logo = f"/content/montage/entree/{n}"; shutil.move(n, R.fichier_logo)
DUREE = M.duree_video(VIDEO)
print(f"✅ Vidéo : {os.path.basename(VIDEO)} ({DUREE:.1f} s)" + (f" — musique : {os.path.basename(MUSIQUE)}" if MUSIQUE else ""))

In [ ]:
#@title 4️⃣ Transcription automatique
mots_bruts = M.transcrire(VIDEO, modele=modele, langue="fr", vocabulaire=vocabulaire)
print("\n📝 Transcription :\n")
print(M.texte_transcription(mots_bruts))

In [ ]:
#@title 5️⃣ Corrections (facultatif) et plan de montage
#@markdown Format : `mot_mal_reconnu=bon_mot; autre=correction` — ex. `Locapé=Locapay; démarcheur=démarcheurs`
corrections = "Locapé=Locapay; Lokapay=Locapay; Locapaï=Locapay" #@param {type:"string"}
mots = M.appliquer_corrections(mots_bruts, corrections)
PLAN = M.construire_plan(mots, DUREE, R)
print(M.resume_plan(PLAN))
print("\nLes mots en MAJUSCULES seront en gras. Relancez cette étape après une correction.")

In [ ]:
#@title 6️⃣ Montage et téléchargement
from IPython.display import Video, display
os.makedirs("/content/montage/sortie", exist_ok=True)
base = os.path.splitext(os.path.basename(VIDEO))[0]
SORTIE = f"/content/montage/sortie/{base}_monte.mp4"
print("🎬 Montage en cours…")
M.monter(VIDEO, PLAN, R, SORTIE, musique=MUSIQUE)
M.ecrire_srt(PLAN, SORTIE.replace(".mp4", ".srt"))
open(SORTIE.replace(".mp4", "_transcription.txt"), "w", encoding="utf-8").write(M.texte_transcription(mots))
print("✅ Terminé !")
display(Video(SORTIE, embed=True, width=320))
files.download(SORTIE)

---
**Fichiers produits** (dossier `/content/montage/sortie`, panneau 📁 à gauche) : la vidéo montée, les sous-titres `.srt` et la transcription `.txt`.

**Astuces**
- Filmez en vertical, la personne au centre, avec un son propre : la transcription n'en sera que meilleure.
- Trop de coupes ? Montez « pause_max_conservee » (étape 2). Pas assez ? Baissez-la.
- Sous-titres qui cachent les mains ou le visage ? Ajustez « position_sous_titres ».